In [0]:
CREATE CATALOG IF NOT EXISTS retail_sales;

CREATE SCHEMA IF NOT EXISTS retail_sales.raw;

CREATE SCHEMA IF NOT EXISTS retail_sales.enriched;

CREATE SCHEMA IF NOT EXISTS retail_sales.analytics;

CREATE SCHEMA IF NOT EXISTS retail_sales.quality;

In [0]:
CREATE EXTERNAL VOLUME IF NOT EXISTS retail_sales.raw.customers_volume
LOCATION 'abfss://external-data-store@companyxyz1207.dfs.core.windows.net/landing/customers/';

CREATE EXTERNAL VOLUME IF NOT EXISTS retail_sales.raw.orders_volume
LOCATION 'abfss://external-data-store@companyxyz1207.dfs.core.windows.net/landing/orders/';

CREATE EXTERNAL VOLUME IF NOT EXISTS retail_sales.raw.products_volume
LOCATION 'abfss://external-data-store@companyxyz1207.dfs.core.windows.net/landing/products/';

CREATE EXTERNAL VOLUME IF NOT EXISTS retail_sales.archive.customers_volume
LOCATION 'abfss://external-data-store@companyxyz1207.dfs.core.windows.net/archive/customers/archive/';

CREATE EXTERNAL VOLUME IF NOT EXISTS retail_sales.archive.orders_volume
LOCATION 'abfss://external-data-store@companyxyz1207.dfs.core.windows.net/archive/orders/archive/';

CREATE EXTERNAL VOLUME IF NOT EXISTS retail_sales.archive.products_volume
LOCATION 'abfss://external-data-store@companyxyz1207.dfs.core.windows.net/archive/products/archive/';

In [0]:
USE CATALOG retail_sales;
USE SCHEMA raw;

In [0]:
SHOW SCHEMAS IN retail_sales;
SHOW VOLUMES IN retail_sales.raw;

In [0]:
-- =========================================================
-- Data Quality Rule Configuration
-- =========================================================

CREATE OR REPLACE TABLE
    retail_sales.quality.data_quality_rules
(
    rule_id            STRING,
    dataset_name       STRING,
    rule_group         STRING,
    rule_name          STRING,
    rule_type          STRING,
    column_name        STRING,
    rule_parameter     STRING,
    failure_reason     STRING,
    rule_order         INT,
    is_active          BOOLEAN
)
USING DELTA;

In [0]:
INSERT INTO retail_sales.quality.data_quality_rules
VALUES

(
    'CUS_001',
    'customers',
    'COMPLETENESS',
    'Customer ID is required',
    'NOT_NULL',
    'customer_id',
    NULL,
    'CUSTOMER_ID_MISSING',
    1,
    TRUE
),

(
    'CUS_002',
    'customers',
    'UNIQUENESS',
    'Customer ID must be unique',
    'UNIQUE',
    'customer_id',
    NULL,
    'DUPLICATE_CUSTOMER_ID',
    2,
    TRUE
),

(
    'CUS_003',
    'customers',
    'VALIDITY',
    'Customer ID format',
    'REGEX',
    'customer_id',
    '^[A-Z]{2}-[0-9]{5}$',
    'CUSTOMER_ID_INVALID',
    3,
    TRUE
),

(
    'CUS_004',
    'customers',
    'COMPLETENESS',
    'Customer name is required',
    'NOT_NULL',
    'customer_name',
    NULL,
    'CUSTOMER_NAME_MISSING',
    4,
    TRUE
),

(
    'CUS_005',
    'customers',
    'VALIDITY',
    'Customer name must contain valid characters',
    'VALID_CUSTOMER_NAME',
    'customer_name',
    NULL,
    'CUSTOMER_NAME_INVALID',
    5,
    TRUE
),

(
    'CUS_006',
    'customers',
    'COMPLETENESS',
    'Email is required',
    'NOT_NULL',
    'email',
    NULL,
    'EMAIL_MISSING',
    6,
    TRUE
),

(
    'CUS_007',
    'customers',
    'VALIDITY',
    'Email format',
    'REGEX',
    'email',
    '^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+[.][A-Za-z]{2,}$',
    'EMAIL_INVALID',
    7,
    TRUE
),

(
    'CUS_008',
    'customers',
    'COMPLETENESS',
    'Customer segment is required',
    'NOT_NULL',
    'segment',
    NULL,
    'CUSTOMER_SEGMENT_MISSING',
    8,
    TRUE
),

(
    'CUS_009',
    'customers',
    'VALIDITY',
    'Customer segment domain',
    'ALLOWED_VALUES',
    'segment',
    '["Consumer","Corporate","Home Office"]',
    'CUSTOMER_SEGMENT_INVALID',
    9,
    TRUE
),

(
    'CUS_010',
    'customers',
    'COMPLETENESS',
    'Customer state is required',
    'NOT_NULL',
    'state',
    NULL,
    'CUSTOMER_STATE_MISSING',
    10,
    TRUE
),

(
    'CUS_011',
    'customers',
    'VALIDITY',
    'Customer state must be valid',
    'VALID_STATE',
    'state',
    NULL,
    'CUSTOMER_STATE_INVALID',
    11,
    TRUE
),

(
    'CUS_012',
    'customers',
    'COMPLETENESS',
    'Country is required',
    'NOT_NULL',
    'country',
    NULL,
    'CUSTOMER_COUNTRY_MISSING',
    12,
    TRUE
);

num_affected_rows,num_inserted_rows
12,12


In [0]:
INSERT INTO retail_sales.quality.data_quality_rules
VALUES

(
    'PRD_001',
    'products',
    'COMPLETENESS',
    'Product ID is required',
    'NOT_NULL',
    'product_id',
    NULL,
    'PRODUCT_ID_MISSING',
    1,
    TRUE
),

(
    'PRD_002',
    'products',
    'UNIQUENESS',
    'Product ID must be unique',
    'UNIQUE',
    'product_id',
    NULL,
    'DUPLICATE_PRODUCT_ID',
    2,
    TRUE
),

(
    'PRD_003',
    'products',
    'VALIDITY',
    'Product ID format',
    'REGEX',
    'product_id',
    '^(FUR|OFF|TEC)-[A-Z]{2}-[0-9]{8}$',
    'PRODUCT_ID_INVALID',
    3,
    TRUE
),

(
    'PRD_004',
    'products',
    'COMPLETENESS',
    'Product price is required',
    'NOT_NULL',
    'price_per_product',
    NULL,
    'PRODUCT_PRICE_MISSING',
    4,
    TRUE
),

(
    'PRD_005',
    'products',
    'VALIDITY',
    'Product price must be positive',
    'POSITIVE',
    'price_per_product',
    NULL,
    'PRODUCT_PRICE_NOT_POSITIVE',
    5,
    TRUE
),

(
    'PRD_006',
    'products',
    'COMPLETENESS',
    'Product category is required',
    'NOT_NULL',
    'category',
    NULL,
    'PRODUCT_CATEGORY_MISSING',
    6,
    TRUE
),

(
    'PRD_007',
    'products',
    'VALIDITY',
    'Product category domain',
    'ALLOWED_VALUES',
    'category',
    '["Furniture","Office Supplies","Technology"]',
    'PRODUCT_CATEGORY_INVALID',
    7,
    TRUE
),

(
    'PRD_008',
    'products',
    'COMPLETENESS',
    'Product subcategory is required',
    'NOT_NULL',
    'sub_category',
    NULL,
    'PRODUCT_SUBCATEGORY_MISSING',
    8,
    TRUE
),

(
    'PRD_009',
    'products',
    'COMPLETENESS',
    'Product name is required',
    'NOT_NULL',
    'product_name',
    NULL,
    'PRODUCT_NAME_MISSING',
    9,
    TRUE
),

(
    'PRD_010',
    'products',
    'COMPLETENESS',
    'Product state is required',
    'NOT_NULL',
    'state',
    NULL,
    'PRODUCT_STATE_MISSING',
    10,
    TRUE
),

(
    'PRD_011',
    'products',
    'VALIDITY',
    'Product state must be valid',
    'VALID_STATE',
    'state',
    NULL,
    'PRODUCT_STATE_INVALID',
    11,
    TRUE
);

num_affected_rows,num_inserted_rows
11,11


In [0]:
INSERT INTO retail_sales.quality.data_quality_rules
VALUES

(
    'ORD_001',
    'orders',
    'COMPLETENESS',
    'Row ID is required',
    'NOT_NULL',
    'row_id',
    NULL,
    'ROW_ID_MISSING',
    1,
    TRUE
),

(
    'ORD_002',
    'orders',
    'UNIQUENESS',
    'Row ID must be unique',
    'UNIQUE',
    'row_id',
    NULL,
    'DUPLICATE_ROW_ID',
    2,
    TRUE
),

(
    'ORD_003',
    'orders',
    'COMPLETENESS',
    'Order ID is required',
    'NOT_NULL',
    'order_id',
    NULL,
    'ORDER_ID_MISSING',
    3,
    TRUE
),

(
    'ORD_004',
    'orders',
    'VALIDITY',
    'Order ID format',
    'REGEX',
    'order_id',
    '^(CA|US)-[0-9]{4}-[0-9]+$',
    'ORDER_ID_INVALID',
    4,
    TRUE
),

(
    'ORD_005',
    'orders',
    'COMPLETENESS',
    'Order date must be present and parseable',
    'NOT_NULL',
    'order_date',
    NULL,
    'ORDER_DATE_INVALID',
    5,
    TRUE
),

(
    'ORD_006',
    'orders',
    'TIMELINESS',
    'Order date cannot be in the future',
    'NOT_FUTURE',
    'order_date',
    NULL,
    'ORDER_DATE_IN_FUTURE',
    6,
    TRUE
),

(
    'ORD_007',
    'orders',
    'COMPLETENESS',
    'Ship date must be present and parseable',
    'NOT_NULL',
    'ship_date',
    NULL,
    'SHIP_DATE_INVALID',
    7,
    TRUE
),

(
    'ORD_008',
    'orders',
    'CONSISTENCY',
    'Ship date must be on or after order date',
    'COLUMN_GTE',
    'ship_date',
    'order_date',
    'SHIP_DATE_BEFORE_ORDER_DATE',
    8,
    TRUE
),

(
    'ORD_009',
    'orders',
    'COMPLETENESS',
    'Customer ID is required',
    'NOT_NULL',
    'customer_id',
    NULL,
    'ORDER_CUSTOMER_ID_MISSING',
    9,
    TRUE
),

(
    'ORD_010',
    'orders',
    'COMPLETENESS',
    'Product ID is required',
    'NOT_NULL',
    'product_id',
    NULL,
    'ORDER_PRODUCT_ID_MISSING',
    10,
    TRUE
),

(
    'ORD_011',
    'orders',
    'COMPLETENESS',
    'Quantity is required',
    'NOT_NULL',
    'quantity',
    NULL,
    'ORDER_QUANTITY_MISSING',
    11,
    TRUE
),

(
    'ORD_012',
    'orders',
    'VALIDITY',
    'Quantity must be positive',
    'POSITIVE',
    'quantity',
    NULL,
    'ORDER_QUANTITY_NOT_POSITIVE',
    12,
    TRUE
),

(
    'ORD_013',
    'orders',
    'COMPLETENESS',
    'Order price is required',
    'NOT_NULL',
    'price',
    NULL,
    'ORDER_PRICE_MISSING',
    13,
    TRUE
),

(
    'ORD_014',
    'orders',
    'VALIDITY',
    'Order price must be positive',
    'POSITIVE',
    'price',
    NULL,
    'ORDER_PRICE_NOT_POSITIVE',
    14,
    TRUE
),

(
    'ORD_015',
    'orders',
    'COMPLETENESS',
    'Discount is required',
    'NOT_NULL',
    'discount',
    NULL,
    'ORDER_DISCOUNT_MISSING',
    15,
    TRUE
),

(
    'ORD_016',
    'orders',
    'VALIDITY',
    'Discount must be between zero and one',
    'BETWEEN',
    'discount',
    '{"min":0,"max":1}',
    'ORDER_DISCOUNT_INVALID',
    16,
    TRUE
),

(
    'ORD_017',
    'orders',
    'COMPLETENESS',
    'Profit must be numeric and non-null',
    'NOT_NULL',
    'profit',
    NULL,
    'ORDER_PROFIT_INVALID',
    17,
    TRUE
),

(
    'ORD_018',
    'orders',
    'COMPLETENESS',
    'Ship mode is required',
    'NOT_NULL',
    'ship_mode',
    NULL,
    'SHIP_MODE_MISSING',
    18,
    TRUE
),

(
    'ORD_019',
    'orders',
    'VALIDITY',
    'Ship mode domain',
    'ALLOWED_VALUES',
    'ship_mode',
    '["Standard Class","Second Class","First Class","Same Day"]',
    'SHIP_MODE_INVALID',
    19,
    TRUE
),

(
    'ORD_020',
    'orders',
    'REFERENTIAL_INTEGRITY',
    'Customer must exist in passed Customers',
    'REFERENCE',
    'customer_id',
    '_customer_reference_valid',
    'CUSTOMER_REFERENCE_INVALID',
    20,
    TRUE
),

(
    'ORD_021',
    'orders',
    'REFERENTIAL_INTEGRITY',
    'Product must exist uniquely in passed Products',
    'REFERENCE',
    'product_id',
    '_product_reference_valid',
    'PRODUCT_REFERENCE_INVALID',
    21,
    TRUE
);

num_affected_rows,num_inserted_rows
21,21


In [0]:
INSERT INTO retail_sales.quality.data_quality_rules
VALUES
(
    'ORD_005A',
    'orders',
    'VALIDITY',
    'Order date must use d/M/yyyy format',
    'DATE_FORMAT',
    'order_date_source',
    'd/M/yyyy',
    'ORDER_DATE_FORMAT_INVALID',
    6,
    TRUE
);

num_affected_rows,num_inserted_rows
1,1


In [0]:
INSERT INTO retail_sales.quality.data_quality_rules
VALUES
(
    'ORD_007A',
    'orders',
    'VALIDITY',
    'Ship date must use d/M/yyyy format',
    'DATE_FORMAT',
    'ship_date_source',
    'd/M/yyyy',
    'SHIP_DATE_FORMAT_INVALID',
    9,
    TRUE
);

num_affected_rows,num_inserted_rows
1,1


In [0]:
-- ==========================================================
-- Additional Customer Data Quality Rules
-- ==========================================================

INSERT INTO retail_sales.quality.data_quality_rules
VALUES

(
    'CUS_013',
    'customers',
    'VALIDITY',
    'Customer country must be United States',
    'ALLOWED_VALUES',
    'country',
    '["United States"]',
    'CUSTOMER_COUNTRY_INVALID',
    13,
    TRUE
),

(
    'CUS_014',
    'customers',
    'COMPLETENESS',
    'Customer address is required',
    'NOT_NULL',
    'address',
    NULL,
    'CUSTOMER_ADDRESS_MISSING',
    14,
    TRUE
),

(
    'CUS_015',
    'customers',
    'COMPLETENESS',
    'Customer city is required',
    'NOT_NULL',
    'city',
    NULL,
    'CUSTOMER_CITY_MISSING',
    15,
    TRUE
),

(
    'CUS_016',
    'customers',
    'COMPLETENESS',
    'Customer postal code is required',
    'NOT_NULL',
    'postal_code',
    NULL,
    'CUSTOMER_POSTAL_CODE_MISSING',
    16,
    TRUE
),

(
    'CUS_017',
    'customers',
    'VALIDITY',
    'Customer postal code must contain five digits',
    'REGEX',
    'postal_code',
    '^[0-9]{5}$',
    'CUSTOMER_POSTAL_CODE_INVALID',
    17,
    TRUE
),

(
    'CUS_018',
    'customers',
    'COMPLETENESS',
    'Customer region is required',
    'NOT_NULL',
    'region',
    NULL,
    'CUSTOMER_REGION_MISSING',
    18,
    TRUE
),

(
    'CUS_019',
    'customers',
    'VALIDITY',
    'Customer region must be valid',
    'ALLOWED_VALUES',
    'region',
    '["West","East","Central","South"]',
    'CUSTOMER_REGION_INVALID',
    19,
    TRUE
);

num_affected_rows,num_inserted_rows
7,7
